In [0]:
from pyspark.sql import functions as F

# ---- Catalog / schema constants ----
CATALOG = "opsanalytics_adb_workspace01"
LAB_SCHEMA = f"{CATALOG}.lab"                 # mapping tables live here
STAGING_SCHEMA = f"{CATALOG}.lab_staging"
PROD_SCHEMA = f"{CATALOG}.lab_production"

SCC_DIR = "/Volumes/opsanalytics_adb_workspace01/lab/raw_data/scc_data"

# ---- PK for the target table ----
SCC_PK = ["ORDERID", "TEST", "RESULTTIME", "TESTNAME"]

# ---- Load mapping tables from the lab schema ----
scc_test_code = spark.table(f"{LAB_SCHEMA}.lab_kpi_scc_test_codes")
scc_setting   = spark.table(f"{LAB_SCHEMA}.lab_kpi_scc_clinictype")
mshs_site     = spark.table(f"{LAB_SCHEMA}.lab_kpi_site_names")
scc_icu_raw   = spark.table(f"{LAB_SCHEMA}.lab_kpi_scc_icu")
tat_targets_raw = spark.table(f"{LAB_SCHEMA}.lab_kpi_turnaround_targets")

# ---- Derived columns on mapping tables (matching the R setup block) ----

# scc_icu: SiteCodeName = paste(SITE, WARD, WARD_NAME)  [R line 142]
scc_icu = scc_icu_raw.withColumn(
    "SiteCodeName",
    F.concat_ws(" ", F.col("SITE"), F.col("WARD"), F.col("WARD_NAME"))
)

# tat_targets: 3-tier Concate  [R lines 128-133]
#   PRIORITY == "All" & PT_SETTING == "All"  -> paste(TEST, DIVISION)
#   PRIORITY != "All" & PT_SETTING == "All"  -> paste(TEST, DIVISION, PRIORITY)
#   else                                     -> paste(TEST, DIVISION, PRIORITY, PT_SETTING)
tat_targets = tat_targets_raw.withColumn(
    "Concate",
    F.when(
        (F.col("PRIORITY") == "All") & (F.col("PT_SETTING") == "All"),
        F.concat_ws(" ", F.col("TEST"), F.col("DIVISION"))
    ).when(
        (F.col("PRIORITY") != "All") & (F.col("PT_SETTING") == "All"),
        F.concat_ws(" ", F.col("TEST"), F.col("DIVISION"), F.col("PRIORITY"))
    ).otherwise(
        F.concat_ws(" ", F.col("TEST"), F.col("DIVISION"), F.col("PRIORITY"), F.col("PT_SETTING"))
    )
)

# ---- Constant lists / orderings (R lines 155-173) ----
CP_MICRO_LAB_ORDER = ["Troponin", "Lactate WB", "BUN", "HGB", "PT", "Rapid Flu", "C. diff"]
ALL_SITES = ["MSH", "MSQ", "MSB", "MSW", "MSM", "MSSN", "RTC"]
HOSP_SITES = ["MSH", "MSQ", "MSB", "MSW", "MSM", "MSSN"]
INFUSION_SITES = ["RTC"]

PT_SETTING_ORDER = ["ED", "ICU", "IP Non-ICU", "Amb", "Other"]
PT_SETTING_ORDER2 = ["ED & ICU", "IP Non-ICU", "Amb", "Other"]
DASHBOARD_PT_SETTING = ["ED & ICU", "IP Non-ICU", "Amb"]
DASHBOARD_PRIORITY_ORDER = ["All", "Stat", "Routine"]
CP_DIVISION_ORDER = ["Chemistry", "Hematology", "Microbiology RRL", "Infusion"]

# ---- Quick verification ----
for name, df in [
    ("scc_test_code", scc_test_code), ("scc_setting", scc_setting),
    ("mshs_site", mshs_site), ("scc_icu", scc_icu), ("tat_targets", tat_targets),
]:
    print(f"{name:16} rows={df.count():>5}  cols={len(df.columns)}")

print("\ntat_targets sample Concate values:")
tat_targets.select("TEST", "DIVISION", "PRIORITY", "PT_SETTING", "Concate").show(5, truncate=False)
print("scc_icu sample SiteCodeName values:")
scc_icu.select("SITE", "WARD", "WARD_NAME", "SiteCodeName").show(5, truncate=False)